In [ ]:
%% #
"""
FTL Syria AI4Climate Hackathon - Group 14
Project Title: Eastern Syria Agro-Climate: Assessing Water Stress & Drought Risks on Rainfed Agriculture
Author: Group 14
Date: September 2026

Description:
This complete script fetches 20 years of climate data for Eastern Syria (Al-Hasakah region) 
from NASA POWER API, calculates the Standardized Precipitation Index (SPI), conducts statistical 
analyses, generates visualisations, and implements a prototype Early-Warning System.
"""

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

# ---------------------------------------------------------
# 1. SETTINGS & STYLING
# ---------------------------------------------------------
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.sans-serif'] = 'Arial'

# Coordinates for Al-Hasakah / Eastern Syria Breadbasket Region
LATITUDE = 36.50
LONGITUDE = 40.75
START_YEAR = 2006
END_YEAR = 2025

print("=" * 70)
print(" FTL Syria AI4Climate - Group 14: Eastern Syria Agro-Climate Analysis")
print("=" * 70)

# ---------------------------------------------------------
# 2. DATA ACQUISITION & PROCESSING (NASA POWER API)
# ---------------------------------------------------------
def fetch_nasa_power_data(lat, lon, start_yr, end_yr):
    """Fetch monthly precipitation and temperature data directly from NASA POWER API."""
    print(f"\n[1/5] Fetching historical climate data (2006-2025) for Eastern Syria (Lat: {lat}, Lon: {lon})...")
    url = (
        f"https://power.larc.nasa.gov/api/temporal/monthly/point?"
        f"parameters=PRECTOTCORR,T2M,T2M_MAX&community=AG&longitude={lon}&latitude={lat}"
        f"&start={start_yr}&end={end_yr}&format=JSON"
    )
    
    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        data = response.json()['properties']['parameter']
        
        # Parse data into DataFrame
        records = []
        for key in data['PRECTOTCORR'].keys():
            # NASA POWER keys are formatted as YYYYMM
            if len(key) == 6 and not key.endswith('13'): # Skip annual 13th month summary
                year = int(key[:4])
                month = int(key[4:])
                precip = data['PRECTOTCORR'][key] # mm/day
                temp = data['T2M'][key]
                temp_max = data['T2M_MAX'][key]
                
                # Convert mm/day to total monthly precipitation (~30 days)
                monthly_precip = precip * 30.44
                records.append({
                    'Year': year,
                    'Month': month,
                    'Precipitation_mm': monthly_precip,
                    'Temp_Avg': temp,
                    'Temp_Max': temp_max
                })
        
        df = pd.DataFrame(records)
        print(" -> Data successfully retrieved from NASA POWER API.")
        return df

    except Exception as e:
        print(f" -> Warning: Could not connect to NASA POWER API ({e}).")
        print(" -> Generating reliable synthetic dataset for demonstration purposes...")
        
        # Fallback synthetic dataset matching Hasakah climate characteristics
        np.random.seed(42)
        records = []
        for yr in range(start_yr, end_yr + 1):
            for mo in range(1, 13):
                # Rainy season (Nov-Apr) vs Dry season (May-Oct)
                is_rainy = mo in [11, 12, 1, 2, 3, 4]
                precip = np.random.uniform(15, 65) if is_rainy else np.random.uniform(0, 5)
                temp = np.random.uniform(6, 18) if is_rainy else np.random.uniform(25, 40)
                records.append({
                    'Year': yr,
                    'Month': mo,
                    'Precipitation_mm': precip,
                    'Temp_Avg': temp,
                    'Temp_Max': temp + np.random.uniform(5, 10)
                })
        return pd.DataFrame(records)

df = fetch_nasa_power_data(LATITUDE, LONGITUDE, START_YEAR, END_YEAR)
# ---------------------------------------------------------
# 3. ANNUAL AGGREGATION & SPI COMPUTATION
# ---------------------------------------------------------
print("\n[2/5] Cleaning and calculating drought indicators (SPI)...")

# Aggregate annually
annual_climate = df.groupby('Year').agg(
    Total_Rain=('Precipitation_mm', 'sum'),
    Mean_Temp=('Temp_Avg', 'mean'),
    Max_Temp=('Temp_Max', 'max')
).reset_index()

# Calculate Standardized Precipitation Index (SPI)
mean_rain = annual_climate['Total_Rain'].mean()
std_rain = annual_climate['Total_Rain'].std()
annual_climate['SPI'] = (annual_climate['Total_Rain'] - mean_rain) / std_rain

# Classify Drought Severity according to SPI thresholds
def classify_drought(spi):
    if spi >= 1.0:
        return 'Wet / رطب'
    elif -1.0 < spi < 1.0:
        return 'Normal / طبيعي'
    elif -1.5 < spi <= -1.0:
        return 'Moderate Drought / جفاف متوسط'
    else:
        return 'Severe Drought / جفاف شديد'

annual_climate['Drought_Status'] = annual_climate['SPI'].apply(classify_drought)

# ---------------------------------------------------------
# 4. STATISTICAL ANALYSES (Hackathon Requirement)
# ---------------------------------------------------------
print("\n[3/5] Performing Statistical Analyses...")
print("-" * 50)

# Analysis 1: Averages, Min/Max
driest_year = annual_climate.loc[annual_climate['Total_Rain'].idxmin()]
hottest_year = annual_climate.loc[annual_climate['Max_Temp'].idxmax()]

print(f"• Average Annual Rainfall (20-yr Mean): {mean_rain:.2f} mm")
print(f"• Driest Year Recorded: {int(driest_year['Year'])} ({driest_year['Total_Rain']:.2f} mm)")
print(f"• Peak Temperature Year: {int(hottest_year['Year'])} ({hottest_year['Max_Temp']:.2f} °C)")

# Analysis 2: Decadal Comparison (% Change)
p1 = annual_climate[annual_climate['Year'] <= 2015]['Total_Rain'].mean()
p2 = annual_climate[annual_climate['Year'] > 2015]['Total_Rain'].mean()
pct_change = ((p2 - p1) / p1) * 100

print(f"• Period 1 (2006-2015) Average Rain: {p1:.2f} mm")
print(f"• Period 2 (2016-2025) Average Rain: {p2:.2f} mm")
print(f"• Rainfall Change Between Decades: {pct_change:+.2f}%")

# Analysis 3: Threshold Detection & Drought Frequency
drought_years = annual_climate[annual_climate['SPI'] <= -1.0]
print(f"• Total Moderate/Severe Drought Years: {len(drought_years)} out of 20 years")

# Analysis 4: Temperature-Rainfall Correlation
corr = annual_climate['Total_Rain'].corr(annual_climate['Mean_Temp'])
print(f"• Correlation between Annual Rain and Mean Temp: {corr:.2f}")

# ---------------------------------------------------------
# 5. VISUALISATION (Main Charts)
# ---------------------------------------------------------
print("\n[4/5] Generating Visualisations...")

fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)

# Subplot 1: Standardized Precipitation Index (SPI) Bar Chart
colors = ['#d9534f' if x <= -1.5 else '#f0ad4e' if x < -1.0 else '#5bc0de' if x < 1.0 else '#5cb85c' for x in annual_climate['SPI']]
axes[0].bar(annual_climate['Year'], annual_climate['SPI'], color=colors, edgecolor='black', alpha=0.85)
axes[0].axhline(0, color='black', linewidth=1)
axes[0].axhline(-1.0, color='red', linestyle='--', label='Drought Threshold (SPI = -1.0)')
axes[0].set_title('Eastern Syria (Al-Hasakah) Standardized Precipitation Index (SPI) [2006 - 2025]', fontsize=13, fontweight='bold')
axes[0].set_ylabel('SPI Value')
axes[0].legend(loc='upper right')
axes[0].grid(True, linestyle=':', alpha=0.6)

# Subplot 2: Rainfall Trend & Max Temperature
ax2 = axes[1].twinx()
line1 = axes[1].plot(annual_climate['Year'], annual_climate['Total_Rain'], color='#1f77b4', marker='o', linewidth=2, label='Total Annual Rain (mm)')
line2 = ax2.plot(annual_climate['Year'], annual_climate['Max_Temp'], color='#d62728', marker='s', linestyle='--', linewidth=2, label='Max Temperature (°C)')

# Combine legends
lines = line1 + line2
labels = [l.get_label() for l in lines]
axes[1].legend(lines, labels, loc='upper right')
axes[1].set_title('Annual Rainfall Trend vs. Peak Summer Temperatures', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Year', fontsize=11)
axes[1].set_ylabel('Rainfall (mm)', color='#1f77b4', fontsize=11)
ax2.set_ylabel('Max Temp (°C)', color='#d62728', fontsize=11)
axes[1].set_xticks(annual_climate['Year'])
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()

# ---------------------------------------------------------
# 6. EARLY-WARNING SYSTEM PROTOTYPE (Solution Output)
# ---------------------------------------------------------
print("\n[5/5] Testing Early-Warning System Framework...")
print("-" * 50)

def evaluate_seasonal_water_stress(current_season_rain, historical_average):
    deficit = ((historical_average - current_season_rain) / historical_average) * 100
    
    if deficit >= 35:
        level = "CRITICAL / حرج"
        action = "High drought risk. Recommend immediate shift to supplementary irrigation & heat-tolerant wheat varieties."
    elif 15 <= deficit < 35:
        level = "WARNING / تحذير"
        action = "Moderate water stress detected. Implement rainwater harvesting and adjust fertilization schedules."
    else:
        level = "NORMAL / طبيعي"
        action = "Optimal moisture levels. Proceed with standard agricultural seasonal plan."
        
    return level, deficit, action

test_rain = 130.0 # Example current seasonal rainfall
level, deficit, action = evaluate_seasonal_water_stress(test_rain, mean_rain)

print(f"• Simulated Season Rainfall: {test_rain:.1f} mm")
print(f"• Historical Baseline Average: {mean_rain:.1f} mm")
print(f"• Calculated Deficit: {deficit:.1f}%")
print(f"• Threat Level: [{level}]")
print(f"• Recommended Response: {action}")

# Chart 2: Visualising the Early Warning System Thresholds
plt.figure(figsize=(10, 5))
scenarios = ['Normal Season', 'Moderate Deficit', 'Simulated Season (130mm)']
rainfall_val = [mean_rain, mean_rain * 0.75, 130.0]
colors_alert = ['#5cb85c', '#f0ad4e', '#d9534f']

bars = plt.bar(scenarios, rainfall_val, color=colors_alert, width=0.5, edgecolor='black')
plt.axhline(mean_rain, color='blue', linestyle='--', label=f'Historical Mean ({mean_rain:.1f} mm)')
plt.axhline(mean_rain * 0.65, color='red', linestyle=':', label='Critical Deficit Threshold (-35%)')

plt.title('Early-Warning Water Stress Assessment (Al-Hasakah Rainfed Agriculture)', fontsize=12, fontweight='bold')
plt.ylabel('Seasonal Rainfall (mm)')
plt.legend(loc='upper right')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 5, f'{yval:.1f} mm', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()

# عرض كل الرسوم البيانية معاً في النهاية
print(" -> Displaying all generated charts...")
plt.show()

print("=" * 70)
print(" Analysis Complete. All deliverables ready for FTL Hackathon Submission.")
print("=" * 70)